# Sentence-level GPT-2 XL surprisal

## Definition

A sentence of words $w_1 \dots w_N$ is tokenized into GPT-2 sub-tokens, with a beginning-of-text token $t_0$ in front.
Word $w_i$ covers sub-tokens $T(w_i)$; every word after the first is encoded with its leading space.

Sub-token surprisal (nats):
$$s(t_k) = -\ln P(t_k \mid t_0, \dots, t_{k-1})$$

Word surprisal, the mean over the word's sub-tokens:
$$S(w_i) = \frac{1}{|T(w_i)|} \sum_{t_k \in T(w_i)} s(t_k)$$

Sentence surprisal, the mean over its words:
$$S(\text{sentence}) = \frac{1}{N} \sum_{i=1}^{N} S(w_i)$$

Each sentence is scored on its own: the context resets at every sentence.

## Setup (once)

```bash
pip install -e ".[lm,notebooks]"
python -c "import nltk; nltk.download('cmudict')"
linguistic fetch gpt2-xl          # about 6 GB, into $LINGUISTIC_RESOURCES (default ~/.cache/linguistic)
```
A GPU is optional. On a CPU, expect about a second per sentence.

## Input

Either point `STIMULI` to a CSV with columns `stimulus_id,condition,text` (`condition` = `sentence`), or type
sentences into `SENTENCES` (this overrides `STIMULI`).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

STIMULI = ROOT / "data/examples/sentences/stimuli.csv"
SENTENCES = []          # e.g. ["The cat sat on the mat.", "Colorless green ideas sleep furiously."]
OUT = ROOT / "out" / "sentence_gpt2xl"

OUT.mkdir(parents=True, exist_ok=True)
if SENTENCES:
    STIMULI = OUT / "stimuli.csv"
    pd.DataFrame({"stimulus_id": [f"s{i + 1:03d}" for i in range(len(SENTENCES))],
                  "condition": "sentence", "text": SENTENCES}).to_csv(STIMULI, index=False)
pd.read_csv(STIMULI).head()

## Step 1: run the pipeline

In [ ]:
from linguistic.cli import _run

_run(STIMULI, OUT, metric_spec=["lm_gpt2-xl"])
words = pd.read_csv(OUT / "words.csv", keep_default_na=False, na_values=[""])
words[["stimulus_id", "word_idx", "word", "lm_n_subtokens_gpt2xl", "lm_surprisal_gpt2xl"]].head(10)

## Step 2: recompute from scratch

The same formulas with `transformers` alone, checked against step 1.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from linguistic import resources
from linguistic.normalize import tokenize

model_dir = resources.path("gpt2-xl").parent
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForCausalLM.from_pretrained(model_dir, dtype=torch.float32).to(device).eval()


def word_surprisals(text):
    ws = tokenize(text)                                    # whitespace split, edge punctuation stripped
    ids, spans = [tok.bos_token_id], []
    for i, w in enumerate(ws):
        piece = tok.encode(w if i == 0 else " " + w)
        spans.append((len(ids), len(ids) + len(piece)))
        ids += piece
    x = torch.tensor([ids], device=device)
    with torch.inference_mode():
        logp = torch.log_softmax(model(x).logits[0].float(), dim=-1)
    s = -logp[:-1].gather(1, x[0, 1:, None])[:, 0].cpu().numpy()   # s[k-1] = surprisal of token k
    return ws, [float(s[a - 1:b - 1].mean()) for a, b in spans]    # S(w_i)


stimuli = pd.read_csv(STIMULI)
rows = []
for r in stimuli.itertuples():
    ws, S = word_surprisals(r.text)
    rows += [{"stimulus_id": r.stimulus_id, "word": w, "surprisal": v} for w, v in zip(ws, S)]
scratch = pd.DataFrame(rows)

diff = np.abs(scratch["surprisal"].to_numpy() - words["lm_surprisal_gpt2xl"].to_numpy())
print(f"{len(scratch)} words; max |scratch - pipeline| = {diff.max():.2e} nats")

## Sentence level

In [ ]:
sent = (scratch.groupby("stimulus_id", sort=False)
             .agg(n_words=("word", "size"), surprisal_gpt2xl=("surprisal", "mean"))
             .join(stimuli.set_index("stimulus_id")["text"]).reset_index())
stim = pd.read_csv(OUT / "stimuli.csv").set_index("stimulus_id")
print("max |scratch - pipeline| (sentences):",
      f'{(sent.set_index("stimulus_id")["surprisal_gpt2xl"] - stim["lm_surprisal_gpt2xl"]).abs().max():.2e}')
sent.sort_values("surprisal_gpt2xl").round(3)

## Examples: highest and lowest surprisal

In [ ]:
def show(df, label):
    print(label)
    for r in df.itertuples():
        print(f"  {r.surprisal_gpt2xl:.2f} nats  {r.text}")

ranked = sent.sort_values("surprisal_gpt2xl", ascending=False)
show(ranked.head(2), "High surprisal")
show(ranked.tail(2)[::-1], "Low surprisal")

Each word counts equally, however many sub-tokens it has. Pooling all sub-tokens of the sentence instead
(their mean surprisal, i.e. log-perplexity) gives more weight to words that split into many sub-tokens.